# PDF → Excel 변환기 (공용차량 운행일지)

**2023-2024년 양식 (폼/모아찍기)** 과 **2025-2026년 양식 (표)** 모두 지원합니다.

셀을 위에서부터 순서대로 `Shift + Enter`로 실행하세요.

## 1단계: 필요한 패키지 설치 (최초 1회만)
아래 셀 실행 후, **Tesseract-OCR**과 **Poppler**도 설치해야 합니다 (스캔 PDF 처리용):
1. **Tesseract**: https://github.com/UB-Mannheim/tesseract/wiki → 설치 시 **Korean** 언어 체크
2. **Poppler**: https://github.com/oschwartz10612/poppler-windows/releases → 압축 해제 후 `bin` 폴더를 PATH에 추가

In [ ]:
!pip install pdfplumber openpyxl pandas tabula-py pytesseract pdf2image

## 2단계: 설정
아래 경로와 옵션을 본인 환경에 맞게 수정하세요.

In [ ]:
#===================================================
# 여기만 수정하세요!
#===================================================

# Tesseract 경로 (설치 경로에 맞게 수정)
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Poppler 경로 (설치 경로에 맞게 수정)
POPPLER_PATH = r"D:\임준호\poppler-25.12.0\Library\bin"

# PDF 파일 또는 폴더 경로
PDF_PATH = r"C:\Users\USER\Documents\txt converter"

# 양식 선택: "auto" / "old" / "new"
FORMAT = "auto"

# 추출 엔진 (new 양식에만): "pdfplumber" / "tabula"
ENGINE = "pdfplumber"

#===================================================

### 설치 확인
아래 셀을 실행하면 Tesseract와 Poppler 경로가 맞는지 자동으로 확인합니다. 오류가 나면 위의 경로를 수정하세요.

In [ ]:
# ====== 경로 확인 (문제가 있으면 2단계 설정을 수정하세요) ======
import shutil, os

print("=" * 50)
print("  설치 확인")
print("=" * 50)

# Tesseract 확인
tesseract_ok = False
tpath = pytesseract.pytesseract.tesseract_cmd
if os.path.isfile(tpath):
    print(f"  Tesseract: {tpath}")
    # 한국어 데이터 확인
    tdir = os.path.dirname(tpath)
    tessdata = os.path.join(os.path.dirname(tdir), "tessdata")
    if not os.path.isdir(tessdata):
        tessdata = os.path.join(tdir, "tessdata")
    kor_file = os.path.join(tessdata, "kor.traineddata")
    if os.path.isfile(kor_file):
        print(f"  한국어 데이터: 설치됨")
        tesseract_ok = True
    else:
        print(f"  한국어 데이터: 미설치!")
        print(f"    → Tesseract 재설치 시 'Korean' 언어를 체크하세요")
        print(f"    → 또는 {tessdata} 폴더에 kor.traineddata 파일을 넣어주세요")
else:
    print(f"  Tesseract: 경로를 찾을 수 없습니다 → {tpath}")
    print(f"    → 2단계에서 tesseract_cmd 경로를 수정하세요")

# Poppler 확인
poppler_ok = False
if POPPLER_PATH:
    pdftotext = os.path.join(POPPLER_PATH, "pdftoppm.exe")
    if not os.path.isfile(pdftotext):
        pdftotext = os.path.join(POPPLER_PATH, "pdftoppm")
    if os.path.isfile(pdftotext):
        print(f"  Poppler: {POPPLER_PATH}")
        poppler_ok = True
    else:
        print(f"  Poppler: 경로에 실행파일이 없습니다 → {POPPLER_PATH}")
        print(f"    → 2단계에서 POPPLER_PATH 경로를 수정하세요")
        # bin 폴더 내용 확인 시도
        if os.path.isdir(POPPLER_PATH):
            files = os.listdir(POPPLER_PATH)
            print(f"    → 폴더 내용: {files[:10]}")
else:
    print(f"  Poppler: 경로 미설정")

print()
if tesseract_ok and poppler_ok:
    print("  모두 정상입니다! 3단계를 실행하세요.")
else:
    print("  위의 문제를 해결한 후 다시 실행하세요.")

## 3단계: 변환 실행
아래 셀을 실행하면 자동으로 변환됩니다.

In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter

# 출력 컬럼
STANDARD_COLUMNS = [
    "날짜", "사용목적", "행선지", "출발시간", "도착시간",
    "운전자", "동승자", "전일누계(km)", "출발(km)", "도착(km)",
    "금일주행(km)", "금일누계(km)",
]


# ========== 스캔 PDF 감지 + OCR ==========
def is_scanned_pdf(pdf_path):
    import pdfplumber
    with pdfplumber.open(pdf_path) as pdf:
        if not pdf.pages: return False
        page = pdf.pages[0]
        text = page.extract_text() or ""
        return len(text.strip()) < 10 and len(page.chars) == 0


def ocr_pdf_to_texts(pdf_path):
    from pdf2image import convert_from_path
    import pytesseract

    print("  OCR 처리 중... (시간이 걸릴 수 있습니다)")

    # 2단계에서 설정한 POPPLER_PATH 사용
    try:
        poppler = POPPLER_PATH
    except NameError:
        poppler = None

    try:
        if poppler:
            images = convert_from_path(pdf_path, dpi=300, poppler_path=poppler)
        else:
            images = convert_from_path(pdf_path, dpi=300)
    except Exception as e:
        print(f"  오류: PDF 이미지 변환 실패 - {e}")
        print("  Poppler 설치를 확인하세요.")
        print("  2단계에서 POPPLER_PATH 경로가 맞는지 확인하세요.")
        if poppler:
            print(f"  현재 POPPLER_PATH: {poppler}")
        return []

    texts = []
    total = len(images)
    for i, img in enumerate(images):
        w, h = img.size
        if w > h * 1.3:  # 가로가 넓으면 모아찍기
            mid = w // 2
            for sub_img in [img.crop((0,0,mid,h)), img.crop((mid,0,w,h))]:
                text = pytesseract.image_to_string(sub_img, lang="kor+eng")
                if text.strip(): texts.append(text)
        else:
            text = pytesseract.image_to_string(img, lang="kor+eng")
            if text.strip(): texts.append(text)
        if (i+1) % 5 == 0 or (i+1) == total:
            print(f"    {i+1}/{total} 페이지 완료...")

    print(f"  OCR 완료: {len(texts)}개 텍스트 추출")
    return texts


def parse_ocr_texts(texts):
    records = []
    for text in texts:
        records.extend(_parse_form(text, []))
    if not records: return pd.DataFrame(columns=STANDARD_COLUMNS)
    return pd.DataFrame(records)


# ========== 양식 자동 감지 ==========
def detect_format(pdf_path):
    import pdfplumber
    with pdfplumber.open(pdf_path) as pdf:
        if not pdf.pages: return "new"
        text = pdf.pages[0].extract_text() or ""
        flat = text.replace(" ", "")
        old_markers = ["차량운행일지", "계기표시", "전일누계", "금일주행", "관리운전원", "금일누계"]
        return "old" if sum(1 for m in old_markers if m in flat) >= 2 else "new"


# ========== 페이지 분할 (모아찍기 지원) ==========
def _split_page(page):
    text = page.extract_text() or ""
    flat = text.replace(" ", "")
    form_count = len(re.findall(r"차량운행일지", flat))
    if form_count >= 2:
        w, h = page.width, page.height
        return [page.crop((0, 0, w/2, h)), page.crop((w/2, 0, w, h))]
    return [page]


# ========== 구 양식 (2023-2024) 파서 ==========
def parse_old_format(pdf_path):
    import pdfplumber
    records = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for sub in _split_page(page):
                text = sub.extract_text() or ""
                if not text.strip(): continue
                tables = sub.extract_tables() or []
                records.extend(_parse_form(text, tables))
    if not records: return pd.DataFrame(columns=STANDARD_COLUMNS)
    return pd.DataFrame(records)


def _parse_form(text, tables):
    records = []

    # 날짜
    date_str = ""
    dm = re.search(r"(\d{4})\s*년\s*(\d{1,2})\s*월\s*(\d{1,2})\s*일", text)
    if dm: date_str = f"{dm.group(1)}-{int(dm.group(2)):02d}-{int(dm.group(3)):02d}"
    if not date_str: return []

    # 사용목적
    purpose = ""
    pm = re.search(r"사용\s*목\s*적\s+(.+)", text)
    if pm:
        purpose = pm.group(1).strip()
        purpose = re.split(r"\s*계기", purpose)[0].strip()

    # 주행거리 요약
    prev_km = today_km = curr_km = ""
    for table in tables:
        for row in table:
            if not row: continue
            row_str = " ".join(str(c) for c in row if c)
            m = re.search(r"전일\s*누계\s*(\d[\d,]*)\s*km", row_str, re.I)
            if m: prev_km = m.group(1).replace(",", "")
            m = re.search(r"금일\s*주행\s*(\d[\d,]*)\s*km", row_str, re.I)
            if m: today_km = m.group(1).replace(",", "")
            m = re.search(r"금일\s*누계\s*(\d[\d,]*)\s*km", row_str, re.I)
            if m: curr_km = m.group(1).replace(",", "")
    # 텍스트 폴백
    if not prev_km:
        m = re.search(r"전일\s*누계\s*(\d[\d,]*)\s*km", text, re.I)
        if m: prev_km = m.group(1).replace(",", "")
    if not today_km:
        m = re.search(r"금일\s*주행\s*(\d[\d,]*)\s*km", text, re.I)
        if m: today_km = m.group(1).replace(",", "")
    if not curr_km:
        m = re.search(r"금일\s*누계\s*(\d[\d,]*)\s*km", text, re.I)
        if m: curr_km = m.group(1).replace(",", "")

    # 운행 건 추출
    trips = _extract_trips(tables, text)

    if trips:
        for i, trip in enumerate(trips):
            records.append({
                "날짜": date_str, "사용목적": purpose,
                "행선지": trip.get("destination", ""),
                "출발시간": trip.get("depart_time", ""),
                "도착시간": trip.get("arrive_time", ""),
                "운전자": trip.get("driver", ""),
                "동승자": trip.get("passenger", ""),
                "전일누계(km)": prev_km if i == 0 else "",
                "출발(km)": trip.get("start_km", ""),
                "도착(km)": trip.get("end_km", ""),
                "금일주행(km)": today_km if i == 0 else "",
                "금일누계(km)": curr_km if i == 0 else "",
            })
    else:
        records.append({
            "날짜": date_str, "사용목적": purpose,
            "행선지": "", "출발시간": "", "도착시간": "",
            "운전자": "", "동승자": "",
            "전일누계(km)": prev_km, "출발(km)": prev_km, "도착(km)": curr_km,
            "금일주행(km)": today_km, "금일누계(km)": curr_km,
        })
    return records


def _extract_trips(tables, text):
    trips = []
    for table in tables:
        for row in table:
            if not row or len(row) < 3: continue
            cells = [str(c).strip() if c else "" for c in row]
            row_text = " ".join(cells)
            dep = re.search(r"출\s*[:\uff1a]\s*(\d{1,2}[:\uff1a]\d{2})", row_text)
            if not dep: continue
            trip = {"depart_time": dep.group(1).replace("\uff1a", ":")}
            arr = re.search(r"착\s*[:\uff1a]\s*(\d{1,2}[:\uff1a]\d{2})", row_text)
            if arr: trip["arrive_time"] = arr.group(1).replace("\uff1a", ":")
            for cell in cells:
                clean = re.sub(r"[출착]\s*[:\uff1a]\s*\d{1,2}[:\uff1a]\d{2}", "", cell)
                nums = re.findall(r"\b(\d{4,6})\b", clean)
                if nums:
                    if "start_km" not in trip:
                        trip["start_km"] = nums[0]
                        if len(nums) >= 2: trip["end_km"] = nums[1]
                    elif "end_km" not in trip:
                        trip["end_km"] = nums[0]
            dest = cells[0].replace("\n", " ").strip() if cells[0] else ""
            skip_pat = ["행선지", "출", "착", "계기", "운행", "운전", "동승", "소속", "성명"]
            if dest and re.search(r"[가-힣]", dest) and not any(s in dest for s in skip_pat):
                trip["destination"] = dest
            if len(cells) >= 6:
                name = cells[5].strip() if len(cells) > 5 else ""
                if name and re.search(r"[가-힣]{2,4}", name) and name not in ("성명", "소속"):
                    trip["driver"] = name
            if len(cells) >= 8:
                pname = cells[7].strip() if len(cells) > 7 else ""
                if pname and re.search(r"[가-힣]{2,4}", pname) and pname not in ("성명", "소속"):
                    trip["passenger"] = pname
            if "driver" not in trip:
                for cell in cells:
                    if re.fullmatch(r"[가-힣]{2,4}", cell):
                        trip["driver"] = cell; break
            trips.append(trip)

    # 텍스트 폴백 (OCR 결과용)
    if not trips:
        for line in text.split("\n"):
            dep = re.search(r"출\s*[:\uff1a]\s*(\d{1,2}[:\uff1a]\d{2})", line)
            if not dep: continue
            trip = {"depart_time": dep.group(1).replace("\uff1a", ":")}
            arr = re.search(r"착\s*[:\uff1a]\s*(\d{1,2}[:\uff1a]\d{2})", line)
            if arr: trip["arrive_time"] = arr.group(1).replace("\uff1a", ":")
            nums = re.findall(r"\b(\d{4,6})\b", re.sub(r"\d{1,2}:\d{2}", "", line))
            if len(nums) >= 2:
                trip["start_km"] = nums[0]; trip["end_km"] = nums[1]
            names = re.findall(r"[가-힣]{2,4}", line)
            skip = {"출발","도착","운행","시간","계기","표시","운전","동승","소속","성명","행선지"}
            for n in names:
                if n not in skip: trip["driver"] = n; break
            trips.append(trip)
    return trips


# ========== 신 양식 (2025-2026) 파서 ==========
def extract_tables_pdfplumber(pdf_path):
    import pdfplumber
    tables = []
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            for table in (page.extract_tables() or []):
                if table:
                    cleaned = [r for r in table if any(c and str(c).strip() for c in r)]
                    if cleaned: tables.append(pd.DataFrame(cleaned))
    return tables

def extract_tables_tabula(pdf_path):
    import tabula
    try: tables = tabula.read_pdf(pdf_path, pages="all", multiple_tables=True, lattice=True)
    except: tables = []
    if not tables:
        try: tables = tabula.read_pdf(pdf_path, pages="all", multiple_tables=True, stream=True)
        except: tables = []
    return [t for t in tables if not t.empty]

def promote_header(df):
    if df.empty: return df
    first_row = df.iloc[0]
    header_like = sum(1 for v in first_row if v and str(v).strip() and not str(v).strip().replace(".","").isdigit())
    if header_like >= len(first_row) * 0.5:
        headers = [str(v).strip() if v else f"Col_{i}" for i, v in enumerate(first_row)]
        df = df.iloc[1:].reset_index(drop=True); df.columns = headers
    return df

def merge_tables(tables):
    if not tables: return pd.DataFrame()
    if len(tables) == 1: return promote_header(tables[0])
    groups = {}
    for t in tables: groups.setdefault(len(t.columns), []).append(t)
    largest = max(groups.values(), key=lambda g: sum(len(t) for t in g))
    norm = []
    for t in largest:
        t = t.copy(); t.columns = range(len(t.columns)); norm.append(t)
    return promote_header(pd.concat(norm, ignore_index=True))


# ========== Excel 스타일링 ==========
def style_excel(wb_path):
    wb = load_workbook(wb_path); ws = wb.active
    hf = Font(name="맑은 고딕", bold=True, size=11, color="FFFFFF")
    hfill = PatternFill(start_color="2F5496", end_color="2F5496", fill_type="solid")
    cf = Font(name="맑은 고딕", size=10)
    border = Border(left=Side(style="thin"), right=Side(style="thin"),
                    top=Side(style="thin"), bottom=Side(style="thin"))
    ca = Alignment(horizontal="center", vertical="center", wrap_text=True)
    la = Alignment(horizontal="left", vertical="center", wrap_text=True)
    for ri, row in enumerate(ws.iter_rows(min_row=1, max_row=ws.max_row, max_col=ws.max_column), 1):
        for cell in row:
            cell.border = border
            if ri == 1: cell.font = hf; cell.fill = hfill; cell.alignment = ca
            else: cell.font = cf; cell.alignment = la
    for ci in range(1, ws.max_column + 1):
        ml = 0; cl = get_column_letter(ci)
        for cell in ws[cl]:
            if cell.value: ml = max(ml, sum(2 if ord(c) > 127 else 1 for c in str(cell.value)))
        ws.column_dimensions[cl].width = min(max(ml + 4, 8), 50)
    ws.freeze_panes = "A2"; wb.save(wb_path)


# ========== 통합 변환 함수 ==========
def convert_pdf_to_excel(pdf_path, output_path=None, engine="pdfplumber", pdf_format="auto"):
    pdf_path = str(Path(pdf_path).resolve())
    if output_path is None:
        output_path = str(Path(pdf_path).with_suffix(".xlsx"))

    # 스캔 PDF 감지 → OCR
    scanned = is_scanned_pdf(pdf_path)
    if scanned:
        print(f"  유형: 스캔(이미지) PDF → OCR 사용")
        texts = ocr_pdf_to_texts(pdf_path)
        if not texts:
            print("  경고: OCR 텍스트 추출 실패"); return None
        df = parse_ocr_texts(texts)
        if df.empty:
            print("  경고: 데이터 파싱 실패"); return None
        print(f"  추출: {len(df)}건")
        df.to_excel(output_path, index=False, sheet_name="운행일지")
        style_excel(output_path)
        return output_path

    # 일반 PDF
    if pdf_format == "auto":
        pdf_format = detect_format(pdf_path)

    fmt_label = "2023-2024 (폼/모아찍기)" if pdf_format == "old" else "2025-2026 (표)"
    print(f"  양식: {fmt_label}")

    if pdf_format == "old":
        df = parse_old_format(pdf_path)
        if df.empty:
            print("  경고: 데이터를 찾지 못했습니다."); return None
        print(f"  추출: {len(df)}건")
    else:
        if engine == "pdfplumber": tables = extract_tables_pdfplumber(pdf_path)
        else: tables = extract_tables_tabula(pdf_path)
        if not tables:
            print("  경고: 테이블을 찾지 못했습니다."); return None
        df = merge_tables(tables)
        print(f"  추출: {len(df)}건")

    df.to_excel(output_path, index=False, sheet_name="운행일지")
    style_excel(output_path)
    return output_path

print("변환 함수 준비 완료! (스캔 PDF OCR + 구/신 양식 + 모아찍기 지원)")

In [ ]:
# 변환 실행
input_path = Path(PDF_PATH)

if input_path.is_dir():
    pdf_files = sorted(input_path.glob("*.pdf")) + sorted(input_path.glob("*.PDF"))
    print(f"폴더에서 PDF {len(pdf_files)}개 발견\n")
    results = []
    for i, pdf_file in enumerate(pdf_files, 1):
        print(f"[{i}/{len(pdf_files)}] {pdf_file.name} 변환 중...")
        result = convert_pdf_to_excel(str(pdf_file), engine=ENGINE, pdf_format=FORMAT)
        if result:
            results.append(result)
            print(f"  → {Path(result).name} 완료!\n")
    print(f"===== 총 {len(results)}개 파일 변환 완료! =====")
elif input_path.is_file():
    print(f"{input_path.name} 변환 중...")
    result = convert_pdf_to_excel(str(input_path), engine=ENGINE, pdf_format=FORMAT)
    if result:
        print(f"\n===== 완료! → {result} =====")
else:
    print(f"경로를 찾을 수 없습니다: {PDF_PATH}")
    print("2단계에서 경로를 다시 확인해주세요.")

## 4단계: 결과 미리보기 (선택사항)
변환된 첫 번째 엑셀 파일의 내용을 확인합니다.

In [ ]:
# 변환된 엑셀 파일 미리보기
if input_path.is_dir():
    xlsx_files = sorted(input_path.glob("*.xlsx"))
    if xlsx_files:
        print(f"미리보기: {xlsx_files[0].name}\n")
        df = pd.read_excel(xlsx_files[0])
        display(df)
    else:
        print("변환된 xlsx 파일이 없습니다.")
elif input_path.is_file():
    xlsx_path = input_path.with_suffix(".xlsx")
    if xlsx_path.exists():
        print(f"미리보기: {xlsx_path.name}\n")
        df = pd.read_excel(xlsx_path)
        display(df)

## [디버그] PDF 내부 텍스트 확인
변환이 안 될 때 아래 셀을 실행해서 결과를 공유해주세요.

In [ ]:
# ====== 디버그: PDF 내부 구조 확인 ======
# 변환 안 되는 PDF 파일 1개의 경로를 넣어주세요
DEBUG_PDF = r"C:\Users\USER\Documents\txt converter\새 문서 (1).pdf"

import pdfplumber

with pdfplumber.open(DEBUG_PDF) as pdf:
    # 첫 페이지만 확인
    page = pdf.pages[0]
    
    print("=" * 60)
    print(f"총 페이지 수: {len(pdf.pages)}")
    print(f"페이지 크기: {page.width} x {page.height}")
    print("=" * 60)
    
    print("\n[1] 추출된 텍스트:")
    print("-" * 60)
    text = page.extract_text()
    if text:
        print(text[:2000])
    else:
        print("(텍스트 없음 - 이미지 기반 PDF일 수 있음)")
    
    print("\n[2] 추출된 테이블:")
    print("-" * 60)
    tables = page.extract_tables()
    if tables:
        for i, table in enumerate(tables):
            print(f"\n--- 테이블 {i+1} ({len(table)}행) ---")
            for row in table[:5]:  # 처음 5행만
                print(row)
            if len(table) > 5:
                print(f"... 외 {len(table)-5}행")
    else:
        print("(테이블 없음)")
    
    print("\n[3] 페이지 내 문자(chars) 수:", len(page.chars))
    if not text and len(page.chars) == 0:
        print("\n⚠ 이 PDF는 이미지 기반(스캔)일 가능성이 높습니다.")
        print("  OCR이 필요할 수 있습니다.")

## [디버그] OCR 테스트 (첫 1페이지만)
위 디버그에서 "이미지 기반 PDF"로 나온 경우, 아래 셀로 OCR이 작동하는지 테스트합니다.

In [ ]:
# ====== OCR 테스트: 첫 1페이지만 빠르게 확인 ======
# 위 DEBUG_PDF 경로와 동일한 파일 사용
from pdf2image import convert_from_path
import pytesseract

print("OCR 테스트 시작...")
print(f"  PDF: {DEBUG_PDF}")
print(f"  POPPLER_PATH: {POPPLER_PATH}")
print(f"  Tesseract: {pytesseract.pytesseract.tesseract_cmd}")
print()

try:
    # 1페이지만 변환
    images = convert_from_path(
        DEBUG_PDF, dpi=300, 
        poppler_path=POPPLER_PATH,
        first_page=1, last_page=1
    )
    print(f"  PDF → 이미지 변환 성공! (이미지 크기: {images[0].size})")
    
    # 모아찍기 확인
    img = images[0]
    w, h = img.size
    is_landscape = w > h * 1.3
    print(f"  가로형(모아찍기): {'예' if is_landscape else '아니오'}")
    
    # OCR 실행
    if is_landscape:
        mid = w // 2
        test_img = img.crop((0, 0, mid, h))
        print("  (왼쪽 반만 OCR 테스트)")
    else:
        test_img = img
    
    ocr_text = pytesseract.image_to_string(test_img, lang="kor+eng")
    
    print(f"\n  OCR 결과 (처음 1000자):")
    print("=" * 60)
    print(ocr_text[:1000])
    print("=" * 60)
    
    if ocr_text.strip():
        print(f"\n  OCR 성공! 총 {len(ocr_text)}자 추출")
        # 날짜 패턴 확인
        import re
        dm = re.search(r"(\d{4})\s*년\s*(\d{1,2})\s*월\s*(\d{1,2})\s*일", ocr_text)
        if dm:
            print(f"  날짜 감지: {dm.group(0)}")
        else:
            print("  경고: 날짜 패턴(YYYY년 MM월 DD일)을 찾지 못했습니다")
            print("  → OCR 품질이 낮거나 양식이 다를 수 있습니다")
    else:
        print("\n  경고: OCR 결과가 비어있습니다!")
        print("  → Tesseract 한국어 데이터(kor.traineddata)가 설치되어 있는지 확인하세요")

except Exception as e:
    print(f"\n  오류 발생: {e}")
    print("  → Poppler/Tesseract 경로를 확인하세요")